In [ ]:

import torch
import psutil

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))

    memory = torch.cuda.get_device_properties(0).total_memory
    print("GPU Memory:", round(memory / (1024**3), 2), "GB")

ram = psutil.virtual_memory().total
print("System RAM:", round(ram / (1024**3), 2), "GB")


PyTorch: 2.14.1+cu130
CUDA: 13.0
GPU Available: True
GPU Name: Tesla T4
GPU Memory: 14.56 GB
System RAM: 12.67 GB


In [ ]:

!apt-get update -qq
!apt-get install -y zstd


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 100 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 1s (964 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 127324 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...


In [ ]:

!curl -fsSL https://ollama.com/install.sh | sh

!ollama --version


>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:

import subprocess
import time
import requests

server = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(5)

response = requests.get("http://127.0.0.1:11434/api/tags")

if response.status_code == 200:
    print("Ollama server started successfully!")
else:
    print("Ollama server is not responding.")


Ollama server started successfully!


In [ ]:
!nvidia-smi

Wed Nov 18 13:40:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:

!ollama pull qwen2.5:1.5b-instruct-q4_K_M


In [ ]:

!ollama pull qwen2.5:1.5b-instruct-q8_0


In [ ]:

!ollama list


NAME                            ID              SIZE      MODIFIED           
qwen2.5:1.5b-instruct-q8_0      b19683a34698    1.6 GB    56 seconds ago        
qwen2.5:1.5b-instruct-q4_K_M    65ec06548149    986 MB    About a minute ago    


In [ ]:

import requests
import time

url = "http://127.0.0.1:11434/api/generate"

data = {
    "model": "qwen2.5:1.5b-instruct-q4_K_M",
    "prompt": "Explain machine learning in simple words.",
    "stream": False,
    "keep_alive": "10m",
    "options": {
        "temperature": 0,
        "num_predict": 80
    }
}

response = requests.post(url, json=data)
result = response.json()

print("Q4 MODEL OUTPUT:")
print(result["response"])

print("\nGeneration tokens:", result.get("eval_count"))
print("Generation time (seconds):",
      round(result.get("eval_duration", 0) / 1e9, 3))

if result.get("eval_duration", 0) > 0:
    speed = result["eval_count"] / (result["eval_duration"] / 1e9)
    print("Tokens per second:", round(speed, 2))


Q4 MODEL OUTPUT:
Machine learning is a type of artificial intelligence that allows computers to learn and improve from experience without being explicitly programmed. It's like giving a computer the ability to learn from data and make decisions or predictions based on that data.

Here are some key points about machine learning:

1. **Learning from Data**: Machine learning algorithms analyze large amounts of data to identify patterns and make predictions. This is often done through

Generation tokens: 80
Generation time (seconds): 23.184
Tokens per second: 3.45


In [ ]:
!ollama ps
!nvidia-smi

NAME                            ID              SIZE      PROCESSOR    CONTEXT    RUNNER      UNTIL              
qwen2.5:1.5b-instruct-q4_K_M    65ec06548149    1.2 GB    100% GPU     4096       llamacpp    9 minutes from now    
Wed Nov 18 13:45:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
|

In [ ]:

import requests
import time
import json
import subprocess

url = "http://127.0.0.1:11434/api/generate"

models = [
    "qwen2.5:1.5b-instruct-q4_K_M",
    "qwen2.5:1.5b-instruct-q8_0"
]

prompts = [
    "Explain machine learning in simple words.",
    "What is the difference between AI and machine learning?",
    "Explain Python lists with a simple example.",
    "What is a database and why is it used?",
    "Explain overfitting in machine learning.",
    "What is an API? Give a simple example.",
    "Explain supervised and unsupervised learning.",
    "Write a Python function to check whether a number is prime.",
    "What is the difference between RAM and ROM?",
    "Explain how a neural network works."
]

def get_gpu_memory():
    result = subprocess.check_output([
        "nvidia-smi",
        "--query-gpu=memory.used",
        "--format=csv,noheader,nounits"
    ])
    return int(result.decode().strip().split("\n")[0])

all_results = {}

for model in models:

    print("\nTesting:", model)

    requests.post(
        url,
        json={"model": model, "keep_alive": 0}
    )

    time.sleep(3)

    start_memory = get_gpu_memory()
    total_tokens = 0
    total_time = 0
    answers = []

    for i, prompt in enumerate(prompts, 1):

        data = {
            "model": model,
            "prompt": prompt,
            "stream": False,
            "keep_alive": "5m",
            "options": {
                "temperature": 0,
                "seed": 42,
                "num_predict": 120,
                "num_ctx": 2048
            }
        }

        response = requests.post(url, json=data)
        result = response.json()

        if response.status_code != 200:
            raise Exception(result)

        tokens = result.get("eval_count", 0)
        seconds = result.get("eval_duration", 0) / 1e9

        total_tokens += tokens
        total_time += seconds

        answers.append({
            "prompt": prompt,
            "answer": result["response"],
            "tokens": tokens,
            "time_seconds": seconds
        })

        print("Completed prompt", i)

    end_memory = get_gpu_memory()

    speed = total_tokens / total_time if total_time > 0 else 0

    all_results[model] = {
        "total_tokens": total_tokens,
        "total_generation_time": total_time,
        "tokens_per_second": speed,
        "gpu_memory_before_mb": start_memory,
        "gpu_memory_after_mb": end_memory,
        "gpu_memory_used_mb": end_memory - start_memory,
        "answers": answers
    }

    print("\nTokens generated:", total_tokens)
    print("Generation time:", round(total_time, 2), "seconds")
    print("Tokens per second:", round(speed, 2))
    print("GPU memory after:", end_memory, "MB")

    requests.post(
        url,
        json={"model": model, "keep_alive": 0}
    )

with open("/content/task3_benchmark.json", "w") as f:
    json.dump(all_results, f, indent=4)

print("\nBoth model benchmarks completed.")
print("Results saved in task3_benchmark.json")



Testing: qwen2.5:1.5b-instruct-q4_K_M
Completed prompt 1
Completed prompt 2
Completed prompt 3
Completed prompt 4
Completed prompt 5
Completed prompt 6
Completed prompt 7
Completed prompt 8
Completed prompt 9
Completed prompt 10

Tokens generated: 1180
Generation time: 8.51 seconds
Tokens per second: 138.74
GPU memory after: 1185 MB

Testing: qwen2.5:1.5b-instruct-q8_0
Completed prompt 1
Completed prompt 2
Completed prompt 3
Completed prompt 4
Completed prompt 5
Completed prompt 6
Completed prompt 7
Completed prompt 8
Completed prompt 9
Completed prompt 10

Tokens generated: 1153
Generation time: 11.43 seconds
Tokens per second: 100.92
GPU memory after: 1815 MB

Both model benchmarks completed.
Results saved in task3_benchmark.json


In [ ]:

import json

with open("/content/task3_benchmark.json", "r") as f:
    results = json.load(f)

q4 = results["qwen2.5:1.5b-instruct-q4_K_M"]["answers"]
q8 = results["qwen2.5:1.5b-instruct-q8_0"]["answers"]

for i in range(10):
    print("\n" + "=" * 70)
    print("QUESTION", i + 1)
    print(q4[i]["prompt"])

    print("\nQ4 ANSWER:")
    print(q4[i]["answer"])

    print("\nQ8 ANSWER:")
    print(q8[i]["answer"])

print("\nANSWER COMPARISON COMPLETED")



QUESTION 1
Explain machine learning in simple words.

Q4 ANSWER:
Machine learning is a type of artificial intelligence that allows computers to learn and improve from experience without being explicitly programmed. It's like giving a computer the ability to learn from data and make decisions or predictions based on that data.

Here are some key points about machine learning:

1. **Learning from Data**: Machine learning algorithms analyze large amounts of data to identify patterns and make predictions. This is often done through supervised learning, where the algorithm is trained on labeled data, or unsupervised learning, where it identifies patterns in unlabeled data.

2. **Making Decisions**: Once the algorithm has learned from

Q8 ANSWER:
Machine learning is a type of artificial intelligence that allows computers to learn and improve from experience without being explicitly programmed. It involves using algorithms and statistical models to analyze data and make predictions or decisi

In [ ]:

for i in range(6):
    print("\n" + "=" * 60)
    print("QUESTION", i + 1)
    print(q4[i]["prompt"])

    print("\nQ4 ANSWER:")
    print(q4[i]["answer"])

    print("\nQ8 ANSWER:")
    print(q8[i]["answer"])



QUESTION 1
Explain machine learning in simple words.

Q4 ANSWER:
Machine learning is a type of artificial intelligence that allows computers to learn and improve from experience without being explicitly programmed. It's like giving a computer the ability to learn from data and make decisions or predictions based on that data.

Here are some key points about machine learning:

1. **Learning from Data**: Machine learning algorithms analyze large amounts of data to identify patterns and make predictions. This is often done through supervised learning, where the algorithm is trained on labeled data, or unsupervised learning, where it identifies patterns in unlabeled data.

2. **Making Decisions**: Once the algorithm has learned from

Q8 ANSWER:
Machine learning is a type of artificial intelligence that allows computers to learn and improve from experience without being explicitly programmed. It involves using algorithms and statistical models to analyze data and make predictions or decisi

In [ ]:

import threading
import time
import subprocess
import requests

def get_gpu_memory():
    result = subprocess.check_output([
        "nvidia-smi",
        "--query-gpu=memory.used",
        "--format=csv,noheader,nounits"
    ])
    return int(result.decode().strip().split()[0])

def benchmark_memory(model):
    requests.post(
        url,
        json={"model": model, "keep_alive": 0}
    )

    time.sleep(3)

    memory_values = []
    stop = False

    def monitor():
        while not stop:
            memory_values.append(get_gpu_memory())
            time.sleep(0.1)

    thread = threading.Thread(target=monitor)
    thread.start()

    for prompt in prompts:
        data = {
            "model": model,
            "prompt": prompt,
            "stream": False,
            "keep_alive": "5m",
            "options": {
                "temperature": 0,
                "num_predict": 120,
                "num_ctx": 2048
            }
        }

        requests.post(url, json=data)

    stop = True
    thread.join()

    return max(memory_values), min(memory_values)

memory_results = {}

for model in models:
    print("\nTesting:", model)

    peak, minimum = benchmark_memory(model)

    memory_results[model] = {
        "peak_gpu_memory_mb": peak,
        "minimum_gpu_memory_mb": minimum
    }

    print("Peak GPU memory:", peak, "MB")
    print("Minimum GPU memory:", minimum, "MB")

with open("/content/task3_memory.json", "w") as f:
    json.dump(memory_results, f, indent=4)

print("\nMemory measurements saved.")



Testing: qwen2.5:1.5b-instruct-q4_K_M
Peak GPU memory: 1185 MB
Minimum GPU memory: 3 MB

Testing: qwen2.5:1.5b-instruct-q8_0
Peak GPU memory: 2997 MB
Minimum GPU memory: 1185 MB

Memory measurements saved.


In [ ]:

import threading
import time
import subprocess
import requests
import json

memory_results = {}

for model in models:
    subprocess.run(
        ["ollama", "stop", model],
        capture_output=True
    )
    time.sleep(5)

    baseline = get_gpu_memory()
    memory_values = []
    stop = False

    def monitor():
        while not stop:
            memory_values.append(get_gpu_memory())
            time.sleep(0.1)

    thread = threading.Thread(target=monitor)
    thread.start()

    for prompt in prompts:
        data = {
            "model": model,
            "prompt": prompt,
            "stream": False,
            "keep_alive": "5m",
            "options": {
                "temperature": 0,
                "num_predict": 120,
                "num_ctx": 2048
            }
        }

        response = requests.post(url, json=data)
        response.raise_for_status()

    stop = True
    thread.join()

    peak = max(memory_values)

    memory_results[model] = {
        "baseline_mb": baseline,
        "peak_gpu_memory_mb": peak,
        "additional_memory_mb": peak - baseline
    }

    subprocess.run(
        ["ollama", "stop", model],
        capture_output=True
    )

    print("\nModel:", model)
    print("Baseline:", baseline, "MB")
    print("Peak:", peak, "MB")
    print("Additional memory:", peak - baseline, "MB")

with open("/content/task3_memory_clean.json", "w") as f:
    json.dump(memory_results, f, indent=4)

print("\nClean memory results saved.")



Model: qwen2.5:1.5b-instruct-q4_K_M
Baseline: 1815 MB
Peak: 2997 MB
Additional memory: 1182 MB

Model: qwen2.5:1.5b-instruct-q8_0
Baseline: 3 MB
Peak: 1815 MB
Additional memory: 1812 MB

Clean memory results saved.


In [ ]:

import json

final_results = {
    "model": "Qwen2.5-1.5B-Instruct",
    "hardware": "NVIDIA Tesla T4 15GB",
    "test_prompts": 10,
    "settings": {
        "temperature": 0,
        "seed": 42,
        "max_new_tokens": 120,
        "context_length": 2048
    },
    "results": {
        "Q4_K_M": {
            "generated_tokens": 1180,
            "generation_time_seconds": 8.51,
            "tokens_per_second": 138.74,
            "additional_gpu_memory_mb": 1182,
            "model_size": "986 MB"
        },
        "Q8_0": {
            "generated_tokens": 1153,
            "generation_time_seconds": 11.43,
            "tokens_per_second": 100.92,
            "additional_gpu_memory_mb": 1812,
            "model_size": "1.6 GB"
        }
    },
    "observations": [
        "Q4 was approximately 37.5% faster.",
        "Q4 used approximately 34.8% less additional GPU memory.",
        "Both models produced broadly comparable answers in manual review."
    ]
}

with open("/content/task3_final_results.json", "w") as f:
    json.dump(final_results, f, indent=4)

print("Final Task 3 results saved successfully!")
print(json.dumps(final_results, indent=4))


Final Task 3 results saved successfully!
{
    "model": "Qwen2.5-1.5B-Instruct",
    "hardware": "NVIDIA Tesla T4 15GB",
    "test_prompts": 10,
    "settings": {
        "temperature": 0,
        "seed": 42,
        "max_new_tokens": 120,
        "context_length": 2048
    },
    "results": {
        "Q4_K_M": {
            "generated_tokens": 1180,
            "generation_time_seconds": 8.51,
            "tokens_per_second": 138.74,
            "additional_gpu_memory_mb": 1182,
            "model_size": "986 MB"
        },
        "Q8_0": {
            "generated_tokens": 1153,
            "generation_time_seconds": 11.43,
            "tokens_per_second": 100.92,
            "additional_gpu_memory_mb": 1812,
            "model_size": "1.6 GB"
        }
    },
    "observations": [
        "Q4 was approximately 37.5% faster.",
        "Q4 used approximately 34.8% less additional GPU memory.",
        "Both models produced broadly comparable answers in manual review."
    ]
}


In [19]:

from google.colab import drive
import os

drive.mount('/content/drive')

folder = "/content/drive/MyDrive"

for root, dirs, files in os.walk(folder):
    dirs[:] = [d for d in dirs if d not in [".ipynb_checkpoints"]]

    for file in files:
        if file.endswith(".ipynb"):
            print(os.path.join(root, file))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/Colab Notebooks/Untitled0.ipynb
/content/drive/MyDrive/Colab Notebooks/FloodShield_360_Data_Preparation.ipynb
/content/drive/MyDrive/Colab Notebooks/Another copy of FloodShield_360_Data_Preparation.ipynb
/content/drive/MyDrive/Colab Notebooks/Copy of FloodShield_360_Data_Preparation.ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled1.ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled2.ipynb
/content/drive/MyDrive/Colab Notebooks/Task2_RAG_From_Scratch.ipynb
/content/drive/MyDrive/Colab Notebooks/Task3 (1).ipynb
/content/drive/MyDrive/Colab Notebooks/Task3.ipynb
/content/drive/MyDrive/Colab Notebooks/Copy of Tasks3.ipynb
/content/drive/MyDrive/Colab Notebooks/	Task1_Fine_Tuning.ipynb
/content/drive/MyDrive/Colab Notebooks/Copy of Task2_RAG_From_Scratch.ipynb
/content/drive/MyDrive/Colab Notebooks/Task3_Local_Inference.ipynb


In [20]:

import json
import os

folder = "/content/drive/MyDrive/Colab Notebooks"

files = [
    "Task1_Fine_Tuning.ipynb",
    "Task2_RAG_From_Scratch.ipynb",
    "Task3_Local_Inference.ipynb"
]

for name in files:
    path = os.path.join(folder, name)

    print("\n" + "=" * 55)
    print("NOTEBOOK:", name)

    if not os.path.exists(path):
        print("NOT FOUND")
        continue

    with open(path, "r", encoding="utf-8") as f:
        nb = json.load(f)

    print("Total cells:", len(nb["cells"]))

    for cell in nb["cells"]:
        if cell["cell_type"] == "markdown":
            content = "".join(cell["source"]).strip()
            if content:
                print("Section:", content[:180].replace("\n", " "))



NOTEBOOK: Task1_Fine_Tuning.ipynb
NOT FOUND

NOTEBOOK: Task2_RAG_From_Scratch.ipynb
Total cells: 33

NOTEBOOK: Task3_Local_Inference.ipynb
Total cells: 18


In [21]:

import os

base = "/content/drive/MyDrive"

print("Searching for Task 1 notebooks...\n")

for root, dirs, files in os.walk(base):
    for name in files:
        if name.lower().endswith(".ipynb") and "task1" in name.lower():
            path = os.path.join(root, name)
            print("Found:", repr(name))
            print("Path:", path)
            print("Size:", round(os.path.getsize(path) / 1024, 2), "KB")
            print()


Searching for Task 1 notebooks...

Found: '\tTask1_Fine_Tuning.ipynb'
Path: /content/drive/MyDrive/Colab Notebooks/	Task1_Fine_Tuning.ipynb
Size: 407.88 KB

